In [ ]:
# ============================================================
# FOIL ALGORITHM - SMALL DATASET
# ============================================================

import math


# ============================================================
# DATASET
# ============================================================

examples = [

    {"Sky": "Sunny",    "Temperature": "Warm",
     "Humidity": "High",   "Play": "No"},

    {"Sky": "Sunny",    "Temperature": "Warm",
     "Humidity": "Normal", "Play": "Yes"},

    {"Sky": "Overcast", "Temperature": "Warm",
     "Humidity": "High",   "Play": "Yes"},

    {"Sky": "Rain",     "Temperature": "Cool",
     "Humidity": "High",   "Play": "Yes"},

    {"Sky": "Rain",     "Temperature": "Cool",
     "Humidity": "Normal", "Play": "Yes"},

    {"Sky": "Rain",     "Temperature": "Warm",
     "Humidity": "Normal", "Play": "No"},

    {"Sky": "Overcast", "Temperature": "Cool",
     "Humidity": "Normal", "Play": "Yes"},

    {"Sky": "Sunny",    "Temperature": "Cool",
     "Humidity": "High",   "Play": "No"}
]


# ============================================================
# COVERED EXAMPLES
# ============================================================

def covered_examples(rule, examples):

    covered = []

    for example in examples:

        satisfies = True

        for feature, value in rule:

            if example[feature] != value:
                satisfies = False
                break

        if satisfies:
            covered.append(example)

    return covered


# ============================================================
# GENERATE CANDIDATE LITERALS
# ============================================================

def generate_candidate_literals(rule, predicates, examples):

    candidates = []

    for feature in predicates:

        values = sorted(
            set(example[feature] for example in examples)
        )

        for value in values:

            literal = (feature, value)

            if literal not in rule:
                candidates.append(literal)

    return candidates


# ============================================================
# FOIL GAIN
# ============================================================

def foil_gain(rule, literal, positives, negatives):

    old_positive = covered_examples(
        rule,
        positives
    )

    old_negative = covered_examples(
        rule,
        negatives
    )

    p0 = len(old_positive)
    n0 = len(old_negative)

    new_rule = rule + [literal]

    new_positive = covered_examples(
        new_rule,
        positives
    )

    new_negative = covered_examples(
        new_rule,
        negatives
    )

    p1 = len(new_positive)
    n1 = len(new_negative)

    if p1 == 0:
        return float("-inf")

    if p0 == 0:
        return float("-inf")

    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)

    return p1 * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )


# ============================================================
# LEARN A NEW RULE
# ============================================================

def learn_new_rule(
    positives,
    negatives,
    predicates,
    examples
):

    new_rule = []

    # NewRuleNeg = Neg
    new_rule_neg = negatives.copy()

    while new_rule_neg:

        candidate_literals = generate_candidate_literals(
            new_rule,
            predicates,
            examples
        )

        if not candidate_literals:
            break

        best_literal = None
        best_gain = float("-inf")

        for literal in candidate_literals:

            gain = foil_gain(
                new_rule,
                literal,
                positives,
                negatives
            )

            if gain > best_gain:

                best_gain = gain
                best_literal = literal

        if best_literal is None:
            break

        # Add BestLiteral to NewRule
        new_rule.append(best_literal)

        # Update NewRuleNeg
        new_rule_neg = covered_examples(
            new_rule,
            negatives
        )

    return new_rule


# ============================================================
# FOIL
# ============================================================

def foil(
    target_predicate,
    target_value,
    predicates,
    examples
):

    # Pos
    pos = [
        example
        for example in examples
        if example[target_predicate] == target_value
    ]

    # Neg
    neg = [
        example
        for example in examples
        if example[target_predicate] != target_value
    ]

    learned_rules = []

    # while Pos
    while pos:

        new_rule = learn_new_rule(
            pos,
            neg,
            predicates,
            examples
        )

        if not new_rule:
            break

        learned_rules.append(new_rule)

        # Remove positive examples covered by NewRule
        covered_pos = covered_examples(
            new_rule,
            pos
        )

        if not covered_pos:
            break

        pos = [
            example
            for example in pos
            if example not in covered_pos
        ]

    return learned_rules


# ============================================================
# PRINT LEARNED RULES
# ============================================================

def print_rules(
    rules,
    target_predicate,
    target_value
):

    for i, rule in enumerate(rules, 1):

        conditions = ", ".join(
            f"{feature}(x,{value})"
            for feature, value in rule
        )

        print(
            f"Rule {i}: "
            f"{target_predicate}(x,{target_value}) :- "
            f"{conditions}"
        )


# ============================================================
# RUN
# ============================================================

predicates = [
    "Sky",
    "Temperature",
    "Humidity"
]

rules = foil(
    target_predicate="Play",
    target_value="Yes",
    predicates=predicates,
    examples=examples
)

print_rules(
    rules,
    "Play",
    "Yes"
)

Rule 1: Play(x,Yes) :- Sky(x,Overcast)
Rule 2: Play(x,Yes) :- Sky(x,Rain), Temperature(x,Cool)
Rule 3: Play(x,Yes) :- Humidity(x,Normal), Sky(x,Sunny)


In [ ]:
# ============================================================
# FOIL ALGORITHM - IRIS DATASET FROM KAGGLE
# ============================================================

import kagglehub
import pandas as pd
import math


# Download Iris dataset directly from Kaggle
path = kagglehub.dataset_download("uciml/iris")

df = pd.read_csv(path + "/Iris.csv")
df = df.drop(columns=["Id"])


# Convert numerical attributes into categorical values
def categorize(x, low, high):
    if x <= low:
        return "LOW"
    elif x <= high:
        return "MEDIUM"
    else:
        return "HIGH"


df["SepalLength"] = df["SepalLengthCm"].apply(
    lambda x: categorize(x, 5.0, 6.0)
)

df["SepalWidth"] = df["SepalWidthCm"].apply(
    lambda x: categorize(x, 3.0, 3.5)
)

df["PetalLength"] = df["PetalLengthCm"].apply(
    lambda x: categorize(x, 2.0, 4.5)
)

df["PetalWidth"] = df["PetalWidthCm"].apply(
    lambda x: categorize(x, 0.5, 1.5)
)

df = df[
    [
        "SepalLength",
        "SepalWidth",
        "PetalLength",
        "PetalWidth",
        "Species"
    ]
]

examples = df.to_dict(orient="records")


# ============================================================
# COVERED EXAMPLES
# ============================================================

def covered_examples(rule, examples):

    covered = []

    for example in examples:

        satisfies = True

        for feature, value in rule:

            if example[feature] != value:
                satisfies = False
                break

        if satisfies:
            covered.append(example)

    return covered


# ============================================================
# GENERATE CANDIDATE LITERALS
# ============================================================

def generate_candidate_literals(rule, predicates, examples):

    candidates = []

    for feature in predicates:

        values = sorted(
            set(example[feature] for example in examples)
        )

        for value in values:

            literal = (feature, value)

            if literal not in rule:
                candidates.append(literal)

    return candidates


# ============================================================
# FOIL GAIN
# ============================================================

def foil_gain(rule, literal, positives, negatives):

    old_rule_positive = covered_examples(
        rule,
        positives
    )

    old_rule_negative = covered_examples(
        rule,
        negatives
    )

    p0 = len(old_rule_positive)
    n0 = len(old_rule_negative)

    new_rule = rule + [literal]

    new_rule_positive = covered_examples(
        new_rule,
        positives
    )

    new_rule_negative = covered_examples(
        new_rule,
        negatives
    )

    p1 = len(new_rule_positive)
    n1 = len(new_rule_negative)

    if p1 == 0:
        return float("-inf")

    if p0 == 0:
        return float("-inf")

    old_probability = p0 / (p0 + n0)
    new_probability = p1 / (p1 + n1)

    gain = p1 * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

    return gain


# ============================================================
# LEARN A NEW RULE
# ============================================================

def learn_new_rule(
    positives,
    negatives,
    predicates,
    examples
):

    # NewRule starts with no preconditions
    new_rule = []

    # NewRuleNeg = Neg
    new_rule_neg = negatives.copy()

    while new_rule_neg:

        candidate_literals = generate_candidate_literals(
            new_rule,
            predicates,
            examples
        )

        if not candidate_literals:
            break

        best_literal = None
        best_gain = float("-inf")

        for literal in candidate_literals:

            gain = foil_gain(
                new_rule,
                literal,
                positives,
                negatives
            )

            if gain > best_gain:

                best_gain = gain
                best_literal = literal

        if best_literal is None:
            break

        # Add BestLiteral to NewRule
        new_rule.append(best_literal)

        # NewRuleNeg = negatives satisfying NewRule
        new_rule_neg = covered_examples(
            new_rule,
            negatives
        )

    return new_rule


# ============================================================
# FOIL
# ============================================================

def foil(
    target_predicate,
    target_value,
    predicates,
    examples
):

    # Pos = examples where target predicate is True
    pos = [
        example
        for example in examples
        if example[target_predicate] == target_value
    ]

    # Neg = examples where target predicate is False
    neg = [
        example
        for example in examples
        if example[target_predicate] != target_value
    ]

    learned_rules = []

    # while Pos
    while pos:

        new_rule = learn_new_rule(
            pos,
            neg,
            predicates,
            examples
        )

        # Cannot learn another rule
        if not new_rule:
            break

        # Add NewRule to Learned_rules
        learned_rules.append(new_rule)

        # Pos = Pos - examples covered by NewRule
        covered_pos = covered_examples(
            new_rule,
            pos
        )

        if not covered_pos:
            break

        pos = [
            example
            for example in pos
            if example not in covered_pos
        ]

    return learned_rules


# ============================================================
# PRINT ONLY LEARNED RULES
# ============================================================

def print_rules(
    rules,
    target_predicate,
    target_value
):

    for i, rule in enumerate(rules, 1):

        if rule:

            conditions = ", ".join(
                f"{feature}(x,{value})"
                for feature, value in rule
            )

            print(
                f"Rule {i}: "
                f"{target_predicate}(x,{target_value}) :- "
                f"{conditions}"
            )

        else:

            print(
                f"Rule {i}: "
                f"{target_predicate}(x,{target_value})"
            )


# ============================================================
# RUN
# ============================================================

predicates = [
    "SepalLength",
    "SepalWidth",
    "PetalLength",
    "PetalWidth"
]

rules = foil(
    target_predicate="Species",
    target_value="Iris-setosa",
    predicates=predicates,
    examples=examples
)

print_rules(
    rules,
    "Species",
    "Iris-setosa"
)

Using Colab cache for faster access to the 'iris' dataset.
Rule 1: Species(x,Iris-setosa) :- PetalLength(x,LOW)
